# L6: LLM API 调用实战

> 掌握 LLM API 的调用方法，理解请求响应格式

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.llm import LLMFactory, Message
    print("✓ LLM 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

## 6.1 LLM API 调用基础

### 什么是 LLM API？

LLM API 是云服务提供商（如 OpenAI、DeepSeek、Anthropic）提供的接口，让我们可以通过 HTTP 请求调用大语言模型。

### 调用流程

```
1. 准备请求：
   - API Key（身份认证）
   - Messages（对话内容）
   - 参数（temperature, max_tokens 等）

2. 发送请求：
   - HTTP POST 到 API 端点
   - 等待响应

3. 解析响应：
   - 提取生成的文本
   - 获取 Token 使用情况
```

### 项目中的 LLM 抽象层

In [ ]:
# 查看项目中支持的 LLM 提供商
from backend.llm.factory import LLMFactory

print("=== 支持的 LLM 提供商 ===\n")
for provider in LLMFactory.list_providers():
    print(f"  - {provider}")

print("\n=== 工厂模式的好处 ===")
print("✓ 统一接口：所有提供商使用相同的 API")
print("✓ 易切换：修改 provider 名称即可更换模型")
print("✓ 易扩展：添加新提供商只需实现 BaseLLM")

## 6.2 调用 DeepSeek API

### DeepSeek 简介

| 特性 | 说明 |
|------|------|
| **公司** | 深度求索 (DeepSeek) |
| **API 兼容** | OpenAI 兼容 |
| **主要模型** | deepseek-chat, deepseek-coder |
| **官网** | https://platform.deepseek.com/ |

### 为什么选择 DeepSeek？

- 性价比高：价格远低于 OpenAI
- 中文友好：针对中文优化
- OpenAI 兼容：可以直接使用 OpenAI SDK

### 练习：调用 DeepSeek API

In [ ]:
import asyncio
from pathlib import Path
from backend.llm import LLMFactory, Message

def get_deepseek_key():
    """获取 DeepSeek API Key"""
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

async def call_deepseek():
    """调用 DeepSeek API"""
    # 获取 API Key
    api_key = get_deepseek_key()
    if not api_key:
        print("⚠️  请在 .env 文件中设置 DEEPSEEK_API_KEY")
        return
    
    # 创建 LLM 实例
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 准备消息
    messages = [
        Message(role="user", content="你好，请用一句话介绍 DeepSeek。")
    ]
    
    print("=== 发送请求到 DeepSeek ===")
    print(f"模型: {llm.model}")
    print(f"消息: {messages[0].content}\n")
    
    # 调用 API
    response = await llm.chat(messages)
    
    print("=== 收到响应 ===")
    print(f"内容: {response.content}")
    print(f"\n=== Token 使用情况 ===")
    print(f"输入 Token: {response.usage.prompt_tokens}")
    print(f"输出 Token: {response.usage.completion_tokens}")
    print(f"总 Token: {response.usage.total_tokens}")
    print(f"模型: {response.model}")
    
    return response

# 运行
await call_deepseek()

## 6.3 多轮对话

### 对话上下文管理

LLM 是**无状态**的，需要我们在每次请求时提供完整的对话历史。

In [ ]:
async def multi_turn_conversation():
    """多轮对话示例"""
    api_key = get_deepseek_key()
    if not api_key:
        print("⚠️  请设置 DEEPSEEK_API_KEY")
        return
    
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 对话历史（初始为空）
    conversation_history = []
    
    # 用户问题
    questions = [
        "什么是 Python？",
        "它有哪些主要特点？",  # 依赖上下文
        "给我一个简单的例子。",  # 依赖上下文
    ]
    
    for i, question in enumerate(questions, 1):
        print(f"\n{'='*50}")
        print(f"第 {i} 轮 - 用户: {question}")
        
        # 添加用户消息到历史
        conversation_history.append(Message(role="user", content=question))
        
        # 调用 API（传入完整历史）
        response = await llm.chat(conversation_history)
        
        print(f"第 {i} 轮 - 助手: {response.content[:100]}...")
        
        # 添加助手回复到历史
        conversation_history.append(
            Message(role="assistant", content=response.content)
        )
    
    print(f"\n{'='*50}")
    print(f"对话结束，共 {len(conversation_history)} 条消息")

await multi_turn_conversation()

## 6.4 参数调优

### Temperature（温度）

Temperature 控制输出的随机性：

| Temperature | 效果 | 适用场景 |
|-------------|------|----------|
| 0 | 确定性，始终相同输出 | 代码生成、数据提取 |
| 0.3 | 低随机性，准确为主 | 技术文档、事实回答 |
| 0.7 | 平衡创意和准确性 | 日常对话、问答 |
| 1.0+ | 高随机性，创意输出 | 创意写作、头脑风暴 |

In [ ]:
async def test_temperature():
    """测试不同 Temperature 的效果"""
    api_key = get_deepseek_key()
    if not api_key:
        print("⚠️  请设置 DEEPSEEK_API_KEY")
        return
    
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    prompt = "给我一个科技公司的创业想法"
    
    temperatures = [0, 0.7, 1.5]
    
    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"Temperature = {temp}")
        print(f"{'='*50}")
        
        response = await llm.chat(
            [Message(role="user", content=prompt)],
            temperature=temp
        )
        
        print(response.content[:200] + "...")

await test_temperature()

### Max Tokens（最大输出长度）

In [ ]:
async def test_max_tokens():
    """测试 max_tokens 参数"""
    api_key = get_deepseek_key()
    if not api_key:
        print("⚠️  请设置 DEEPSEEK_API_KEY")
        return
    
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    prompt = "请详细介绍 Python 编程语言的历史、特点和应用领域"
    
    # 限制输出长度
    response = await llm.chat(
        [Message(role="user", content=prompt)],
        max_tokens=100  # 只允许最多 100 个输出 token
    )
    
    print("=== 限制 max_tokens=100 的输出 ===\n")
    print(response.content)
    print(f"\n实际输出 Token 数: {response.usage.completion_tokens}")

await test_max_tokens()

## 6.5 查看请求/响应格式

### 理解 API 数据格式

In [ ]:
import json

def print_schema(model_class):
    """格式化打印 Pydantic 模型的 JSON Schema"""
    print(json.dumps(model_class.model_json_schema(), indent=2, ensure_ascii=False))

# 展示 Message 模型
print("=== Message 模型 ===")
print_schema(Message)

print("\n=== TokenUsage 模型 ===")
print_schema(TokenUsage)

print("\n=== ChatResponse 模型 ===")
print_schema(ChatResponse)

### 原始 API 请求/响应示例

In [ ]:
import json

# 展示 DeepSeek API 的请求格式
request_example = {
    "model": "deepseek-chat",
    "messages": [
        {"role": "system", "content": "你是一个有用的助手。"},
        {"role": "user", "content": "你好"}
    ],
    "temperature": 0.7,
    "max_tokens": 1000
}

response_example = {
    "id": "chat-123456789",
    "object": "chat.completion",
    "created": 1234567890,
    "model": "deepseek-chat",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "你好！有什么我可以帮助你的吗？"
            },
            "finish_reason": "stop"
        }
    ],
    "usage": {
        "prompt_tokens": 10,
        "completion_tokens": 8,
        "total_tokens": 18
    }
}

print("=== DeepSeek API 请求格式 ===\n")
print(json.dumps(request_example, indent=2, ensure_ascii=False))

print("\n=== DeepSeek API 响应格式 ===\n")
print(json.dumps(response_example, indent=2, ensure_ascii=False))

## 6.6 项目源码解析

### 查看 DeepSeek 客户端实现

In [ ]:
import inspect
from backend.llm.deepseek_client import DeepSeekLLM

# 查看 chat 方法的签名
print("=== DeepSeekLLM.chat 方法签名 ===\n")
print(inspect.signature(DeepSeekLLM.chat))

# 查看源代码
print("\n=== DeepSeekLLM.chat 源代码（部分）===\n")
source = inspect.getsource(DeepSeekLLM.chat)
print(source[:800] + "...")

## 6.7 错误处理

### 常见错误类型

In [ ]:
from backend.llm.base import (
    LLMError,
    LLMRateLimitError,
    LLMTimeoutError,
    LLMAuthenticationError,
    LLMInvalidRequestError
)

print("=== LLM 错误类型 ===\n")
print(f"LLMError: 基础错误类")
print(f"  ├─ LLMRateLimitError: 速率限制（请求过于频繁）")
print(f"  ├─ LLMTimeoutError: 请求超时")
print(f"  ├─ LLMAuthenticationError: 认证失败（API Key 错误）")
print(f"  └─ LLMInvalidRequestError: 无效请求（参数错误）")

### 错误处理示例

In [ ]:
async def error_handling_demo():
    """错误处理示例"""
    # 模拟使用无效 API Key
    llm = LLMFactory.create("deepseek", api_key="invalid_key")
    
    try:
        response = await llm.chat([
            Message(role="user", content="你好")
        ])
    except LLMAuthenticationError as e:
        print(f"认证错误: {e.message}")
        print(f"提供商: {e.provider}")
        print(f"详情: {e.details}")
    except LLMRateLimitError as e:
        print(f"速率限制: {e.message}")
        print("建议: 等待一段时间后重试")
    except LLMTimeoutError as e:
        print(f"请求超时: {e.message}")
        print("建议: 检查网络连接或增加超时时间")
    except LLMError as e:
        print(f"LLM 错误: {e.message}")

# 运行（取消注释以测试错误处理）
# await error_handling_demo()

## 6.8 综合练习

### 练习：构建一个简单的聊天机器人

In [ ]:
class SimpleChatbot:
    """简单的聊天机器人"""
    
    def __init__(self, provider="deepseek", api_key=None):
        """初始化聊天机器人"""
        if api_key is None:
            api_key = get_deepseek_key()
        
        self.llm = LLMFactory.create(provider, api_key=api_key)
        self.history = []
        self.system_prompt = "你是一个友好、专业的AI助手。"
    
    async def chat(self, user_message, temperature=0.7):
        """与用户对话"""
        # 构建消息列表
        messages = [Message(role="system", content=self.system_prompt)]
        messages.extend(self.history)
        messages.append(Message(role="user", content=user_message))
        
        # 调用 LLM
        response = await self.llm.chat(messages, temperature=temperature)
        
        # 更新历史
        self.history.append(Message(role="user", content=user_message))
        self.history.append(Message(role="assistant", content=response.content))
        
        # 限制历史长度（避免超出 context window）
        if len(self.history) > 20:
            self.history = self.history[-20:]
        
        return response.content
    
    def clear_history(self):
        """清空对话历史"""
        self.history = []
    
    def get_token_count(self):
        """估算对话历史的 token 数量"""
        # 粗略估算：中文约 2 字符/token，英文约 4 字符/token
        total_chars = sum(len(m.content) for m in self.history)
        return total_chars // 3  # 粗略估算

# 使用示例
async def chatbot_demo():
    """聊天机器人演示"""
    bot = SimpleChatbot()
    
    # 模拟对话
    questions = [
        "你好，我叫小明",
        "我刚才告诉你我叫什么名字？",
        "请给我推荐一本 Python 入门书籍"
    ]
    
    for q in questions:
        print(f"\n用户: {q}")
        answer = await bot.chat(q)
        print(f"助手: {answer[:100]}...")
        print(f"[估算 Token 数: {bot.get_token_count()}]")

# 运行
await chatbot_demo()

## 6.9 常见面试问题

**Q: LLM API 调用的基本流程是什么？**
- A:
  1. 准备 API Key 和请求参数
  2. 构建 Messages 数组
  3. 发送 HTTP POST 请求
  4. 解析响应，提取内容和 usage
  5. 处理可能的错误

**Q: 为什么 LLM 是无状态的？如何管理对话历史？**
- A:
  - LLM API 是无状态的，每次请求独立
  - 需要客户端维护对话历史（messages 数组）
  - 每次请求携带完整历史
  - 注意 context window 限制，需要裁剪历史

**Q: Temperature 参数如何影响输出？**
- A:
  - 0: 确定性输出，相同输入总是相同输出
  - 0.7: 平衡随机性和准确性
  - 1.0+: 高随机性，适合创意任务
  - 原理：控制采样时的概率分布平滑程度

**Q: 如何优化 API 调用成本？**
- A:
  1. 精简 Prompt，减少输入 Token
  2. 设置合理的 max_tokens
  3. 使用缓存（对重复问题）
  4. 选择性价比高的模型（如 DeepSeek）
  5. 批量处理请求

**Q: 如何处理 API 速率限制（Rate Limit）？**
- A:
  1. 捕获 LLMRateLimitError 异常
  2. 实现指数退避重试
  3. 使用请求队列控制并发
  4. 考虑升级 API 套餐

**Q: 项目中的 LLMFactory 工厂模式有什么好处？**
- A:
  - 解耦：客户端代码不依赖具体实现
  - 易扩展：添加新提供商只需注册
  - 易切换：修改配置即可更换模型
  - 统一接口：所有提供商使用相同 API

**Q: Message 中的 role 字段有哪些取值？**
- A:
  | role | 说明 |
  |------|------|
  | system | 系统提示，设定 AI 行为 |
  | user | 用户消息 |
  | assistant | AI 的回复 |
  | tool | 工具调用的返回结果 |

---

## 总结

本课程学习了 LLM API 调用的核心知识：

| 知识点 | 说明 |
|--------|------|
| **API 调用流程** | 准备请求 → 发送 → 解析响应 |
| **消息格式** | system/user/assistant 三种角色 |
| **参数控制** | temperature、max_tokens |
| **对话历史** | 需要客户端维护和管理 |
| **错误处理** | 认证、速率限制、超时 |
| **工厂模式** | LLMFactory 统一多提供商接口 |

**下一步**: L7 将深入讲解多模型抽象设计！